In [1]:
import numpy as np
import matplotlib.pyplot as plt

class BanditEnv:
    """
    A k-armed Gaussian Bandit.
    True action values are drawn once at constuction (__init__).
    Rewards are sampled from N(q*(a), 1) on each call to step()
    """
    def __init__(self, k=10, seed=None):
        self.k = k
        self.rng = np.random.RandomState(seed)

        self.true_values = self.rng.randn(k)
        self.optimal = int(np.argmax(self.true_values))

    def step(self, action):
        mean = self.true_values[action]
        return self.rng.normal(mean, 1)

In [2]:
env = BanditEnv(k=10, seed=42)
print("True values:", env.true_values.round(2))
print("Optimal arm:", env.optimal)
print("Sample reward:", env.step(env.optimal))

True values: [ 0.5  -0.14  0.65  1.52 -0.23 -0.23  1.58  0.77 -0.47  0.54]
Optimal arm: 6
Sample reward: 1.1157951226949292


In [3]:
def random_policy(Q, N, t):
    return np.random.randint(len(Q))

def greedy_policy(Q, N, t):
    candidates = np.where(Q == np.max(Q))[0]
    return int(np.random.choice(candidates))

def epsilon_greedy_policy(Q, N, t, epsilon=0.1):
    if np.random.random() < epsilon:
        return np.random.randint(len(Q))
    else:
        return greedy_policy(Q, N, t)
        
def eps_greedy_10(Q, N, t):
    return epsilon_greedy_policy(Q, N, t, epsilon=0.10)
    
def eps_greedy_01(Q, N, t):
    return epsilon_greedy_policy(Q, N, t, epsilon=0.01)

In [4]:
def run_bandit(env, policy, num_rounds=1000):
    """
    Run one bandit experiment for num_rounds rounds
    Returns: rewards, actions, Q (final estimates), and N (final counts)
    """

    k = env.k
    Q = np.zeros(k)
    N = np.zeros(k, dtype=int)

    rewards = []
    actions = []

    for t in range(num_rounds):
        action = policy(Q, N, t)
        reward = env.step(action)
        N[action] += 1
        Q[action] += (reward - Q[action]) / N[action]

        rewards.append(reward)
        actions.append(action)

    return rewards, actions, Q, N

In [5]:
env = BanditEnv(k=10, seed=42)
rewards, actions, Q, N = run_bandit(env, eps_greedy_10, num_rounds=1000)
print("Q estimates:", Q.round(2))
print("True values:", env.true_values.round(2))
print("Pull counts:", N)
print("Optimal arm:", env.optimal, "  Most-pulled:", np.argmax(N))

Q estimates: [ 0.37 -0.62  0.29  1.25 -0.37  0.41  1.61  0.74 -0.03  0.06]
True values: [ 0.5  -0.14  0.65  1.52 -0.23 -0.23  1.58  0.77 -0.47  0.54]
Pull counts: [ 16  12  13   7  16   7 895  13  12   9]
Optimal arm: 6   Most-pulled: 6


In [6]:
def average_reward_curve(rewards):
    return np.cumsum(rewards) / np.arange(1, len(rewards) + 1)

def percent_optimal_curve(actions, optimal_arm):
    """
    Running % of times the optimal arm was chosen
    Returns an array of length len(actions), values in [0, 100]
    """
    is_optimal = [1 if a == optimal_arm else 0 for a in actions]
    return (np.cumsum(is_optimal) / np.arange(1, len(actions) + 1)) * 100

def compute_cumulative_regret(rewards, optimal_value):
    T = len(rewards)
    return np.arange(1, T + 1) * optimal_value - np.cumsum(rewards) 

In [7]:
env = BanditEnv(k=10, seed=0)
rewards_r, actions_r, _, _ = run_bandit(env, random_policy, 1000)
pct = percent_optimal_curve(actions_r, env.optimal)
reg = compute_cumulative_regret(rewards_r, env.true_values[env.optimal])
print(f"Random: final % optimal = {pct[-1]:.1f}%, (expect ~10%)")
print(f"Random: final regret = {reg[-1]:.1f}")

Random: final % optimal = 8.5%, (expect ~10%)
Random: final regret = 1543.8


In [ ]:
def run_experiment(seeds, policy, num_rounds=1000):
    all_avg_rewards = []
    all_pct_optimal = []
    all_regrets = []

    for seed in seeds:
        env = BanditEnv(k=10, seed=seed)
        optimal_arm = env.optimal
        optimal_value = env.true_values[optimal_arm]

        rewards, actions, _, _ = run_bandit(env, policy, num_rounds)

        all_avg_rewards.append(average_reward_curve(rewards))
        all_pct_optimal.append(percent_optimal_curve(actions, optimal_arm))
        all_regrets.append(compute_cumulative_regret(rewards, optimal_value))

    return {
        "average_reward" : np.mean(all_avg_rewards, axis=0),
        "percent_optimal" : np.mean(all_pct_optimal, axis=0),
        "cumulative_regret" : np.mean(all_regrets, axis=0),
    }

num_runs = 200
num_rounds = 1000
seeds = list(range(num_runs))

random_results = run_experiment(seeds, random_policy, num_rounds)
greedy_results = run_experiment(seeds, greedy_policy, num_rounds)
eps10_results = run_experiment(seeds, eps_greedy_10, num_rounds)
eps01_results = run_experiment(seeds, eps_greedy_01, num_rounds)


print()
print(f"{'Strategy':<35} {'Avg reward':>12} {'% Optimal':>11} {'Regret':>10}")
print("-" * 72)
for name, res in [
    ("Random",                       random_results),
    ("Greedy",                       greedy_results),
    ("eps-greedy (eps=0.10)",        eps10_results),
    ("eps-greedy (eps=0.01)",        eps01_results),
]:
    avg_rew = res["average_reward"][-1]
    pct_opt = res["percent_optimal"][-1]
    regret  = res["cumulative_regret"][-1]
    print(f"{name:<35} {avg_rew:>12.3f} {pct_opt:>10.1f}% {regret:>10.1f}")
print()


In [ ]:
def plot_results(results_dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, res in results_dict.items():
        axes[0].plot(res["average_reward"], label=name)
        axes[1].plot(res["percent_optimal"], label=name)
    axes[0].set_xlabel("Round"); axes[0].set_ylabel("Average reward")
    axes[0].set_title("Average reward per round"); axes[0].legend(fontsize=9)
    axes[1].set_xlabel("Round"); axes[1].set_ylabel("% Optimal action")
    axes[1].set_title("Percentage of optimal arm chosen"); axes[1].legend(fontsize=9)
    plt.tight_layout()
    plt.show()

results_dict = {
    "Random": random_results,
    "Greedy": greedy_results,
    "eps-greedy (eps=0.10)": eps10_results,
    "eps-greedy (eps=0.01)": eps01_results,
}
plot_results(results_dict)